# Main fine-tuning and evaluation workflow

Train candidate checkpoints on the saved train pairs and evaluate them on validation. Export the checkpoint selected by its measured validation MRR@10. Preserve input hashes, actual hardware details and observed training logs with each experiment.

The weight-decay ablation still requires actual controlled runs. Current measured CPU results are documented in `reports/e07_e08_finetune_main.md`; registry promotion is a separate reviewed action.


## 1. Hardware Detection & Real GPU Environment (DoD a)

In compliance with DoD, we detect and record the actual GPU hardware without simulation or placeholder values.

In [ ]:
import os, sys, platform, time, torch

print("=== System & Hardware Environment ===")
print(f"OS Platform     : {platform.platform()}")
print(f"Python Version  : {sys.version.split()[0]}")
print(f"CUDA Available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device Name : {gpu_name}")
    print(f"Total GPU Memory: {gpu_mem:.2f} GB")
else:
    print("WARNING: No GPU detected. Running on CPU with accelerated settings.")

## 2. Setup Repository & Dependencies

Clone the repository and install required packages (`sentence-transformers`, `mlflow`, `ranx`, `matplotlib`).

In [ ]:
# In Colab, clone the repo if not present:
REPO_URL = "https://github.com/minhhoangb/food-recipe-recommendation-system.git"
BRANCH = "dev"

if not os.path.exists("src"):
    !git clone --branch {BRANCH} {REPO_URL} /content/repo
    %cd /content/repo

!pip install -q -r requirements.txt
!pip install -q mlflow sentence-transformers matplotlib
print("Dependencies successfully installed.")

## 3. Generate ~30,000 Training Pairs for Main Profile (DoD a)

- Profile: `main` (`configs/default.yaml`).
- Split: `train` only (strictly no validation/test leakage).
- Query format: Ingredients permutation with multiple seeds via `src.queries.make_query_ingredients`.
- Positive format: Same formatted document string used by Hưng (`src.embeddings.format_document`).
- Max pairs: 30,000, seed: 42. SHA-256 checksum recorded in manifest.

In [ ]:
# Generate pairs using the modular build_pairs pipeline
!python -m src.build_pairs --config configs/default.yaml --split train --mode ingredients --max-pairs 30000 --seed 42

import json
from pathlib import Path

manifest_path = Path("data/pairs/main_train_ingredients_pairs_manifest.json")
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("=== Pairs Manifest Information ===")
    print(f"File Name   : {manifest.get('file')}")
    print(f"Pair Count  : {manifest.get('pair_count'):,}")
    print(f"Seed        : {manifest.get('seed')}")
    print(f"SHA-256 Hash: {manifest.get('sha256')}")

## 4. Prepare Training Dataset & Validation Evaluator (DoD a)

Prepare `Dataset` from pairs and initialize `InformationRetrievalEvaluator` on `data/processed/main/` validation queries and corpus.

In [ ]:
from datasets import Dataset
from src.build_pairs import load_pairs
from src.utils import read_jsonl
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# Load training pairs
pairs_file = "data/pairs/main_train_ingredients_pairs.jsonl"
train_pairs = load_pairs(pairs_file)
train_dataset = Dataset.from_dict({
    "query": [p["query"] for p in train_pairs],
    "positive": [p["positive"] for p in train_pairs],
})
print(f"Loaded {len(train_dataset):,} pairs into HuggingFace Dataset.")

# Setup validation evaluator on main profile
val_queries_raw = [r for r in read_jsonl("data/processed/main/queries.jsonl") if r.get("split") == "validation"]
val_corpus_raw = list(read_jsonl("data/processed/main/corpus.jsonl"))
val_qrels_raw = list(read_jsonl("data/processed/main/qrels.jsonl"))

val_queries = {q["query_id"]: ", ".join(q["ingredients"]) for q in val_queries_raw}
val_corpus = {c["recipe_id"]: ", ".join(c["ingredients_normalized"]) for c in val_corpus_raw}
val_relevant_docs = {}
for row in val_qrels_raw:
    if row["query_id"] in val_queries and int(row["relevance"]) >= 1:
        val_relevant_docs.setdefault(row["query_id"], set()).add(row["recipe_id"])

evaluator = InformationRetrievalEvaluator(
    queries=val_queries,
    corpus=val_corpus,
    relevant_docs=val_relevant_docs,
    name="validation_main",
    mrr_at_k=[10],
    ndcg_at_k=[10],
    accuracy_at_k=[5, 10],
    show_progress_bar=False,
)
print(f"Evaluator initialized with {len(val_queries)} validation queries over {len(val_corpus)} candidate recipes.")

## 5. Main Fine-Tuning: 2 Configurations (DoD a)

We train 2 standard configurations for 3 epochs with `load_best_model_at_end=True` choosing the optimal checkpoint according to validation MRR@10:
- **Config 1:** `learning_rate=2e-5`, `batch_size=32`, `weight_decay=0.01`
- **Config 2:** `learning_rate=1e-5`, `batch_size=64`, `weight_decay=0.01`

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers
import mlflow

BASE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

configs = [
    {"name": "config1_lr2e-5_b32", "lr": 2e-5, "batch_size": 32, "wd": 0.01},
    {"name": "config2_lr1e-5_b64", "lr": 1e-5, "batch_size": 64, "wd": 0.01},
]

run_results = {}

for cfg in configs:
    print(f"\n>>> Starting Fine-Tuning: {cfg['name']} <<<")
    start_time = time.time()
    
    model = SentenceTransformer(BASE_MODEL_NAME)
    model.max_seq_length = 128
    loss_func = losses.MultipleNegativesRankingLoss(model)
    
    output_dir = f"checkpoints/{cfg['name']}"
    
    training_args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=cfg["batch_size"],
        learning_rate=cfg["lr"],
        warmup_ratio=0.1,
        weight_decay=cfg["wd"],
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_validation_main_cosine_mrr@10",
        greater_is_better=True,
        logging_steps=50,
        seed=42,
    )
    
    trainer = SentenceTransformerTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=None,
        loss=loss_func,
        evaluator=evaluator,
    )
    
    train_output = trainer.train()
    elapsed = time.time() - start_time
    
    eval_metrics = evaluator(model)
    val_mrr = eval_metrics.get("validation_main_cosine_mrr@10", 0.0)
    val_recall = eval_metrics.get("validation_main_cosine_accuracy@10", 0.0)
    
    print(f"Completed {cfg['name']} in {elapsed:.1f}s | Val MRR@10: {val_mrr:.4f} | Recall@10: {val_recall:.4f}")
    run_results[cfg['name']] = {
        "model": model,
        "trainer": trainer,
        "time_sec": elapsed,
        "val_mrr@10": val_mrr,
        "val_recall@10": val_recall,
    }

## 6. Weight-decay ablation status

A controlled comparison still needs two actual runs differing only in weight decay, with the same seed, data and schedule. Plot the recorded trainer logs and evaluator results after those runs finish. No measured ablation result or loss curve is available in this notebook.


## 7. Inspect existing MLflow runs

Read the runs produced by actual training and evaluation. Open the real UI with `mlflow ui --backend-store-uri sqlite:///mlflow.db`. Registry aliases are reviewed separately from this read-only inspection.


In [ ]:
from pathlib import Path
from mlflow.tracking import MlflowClient

if not Path("mlflow.db").is_file():
    raise RuntimeError("No tracking database found; run actual training or evaluation first")

client = MlflowClient(tracking_uri="sqlite:///mlflow.db")
for experiment in client.search_experiments():
    print(f"Experiment: {experiment.name}")
    for run in client.search_runs(experiment_ids=[experiment.experiment_id]):
        print(run.info.run_id, run.info.status, run.data.metrics)


## 8. Export the best measured checkpoint

Run the training cells first. Export only a completed training result selected by validation MRR@10, then compute its weight hash for the dense bundle hand-off.


In [ ]:
import hashlib

from pathlib import Path

if not globals().get("run_results"):
    raise RuntimeError("No completed training results; run the training cells before export")
best_result = max(run_results.values(), key=lambda result: result["val_mrr@10"])
best_model = best_result["model"]

checkpoint_export_dir = Path("models/minilm_finetuned_main")
checkpoint_export_dir.mkdir(parents=True, exist_ok=True)
best_model.save_pretrained(str(checkpoint_export_dir))
print(f"Best checkpoint exported to: {checkpoint_export_dir}")

# Compute SHA-256 of model weights
weights_file = checkpoint_export_dir / "model.safetensors"
if not weights_file.exists():
    weights_file = checkpoint_export_dir / "pytorch_model.bin"

if weights_file.exists():
    h = hashlib.sha256()
    h.update(weights_file.read_bytes())
    sha256_val = h.hexdigest()
    print(f"Model Weights File: {weights_file.name}")
    print(f"SHA-256 Checksum : {sha256_val}")

print("\n=== Command for Hưng (Person 2) to build the dense bundle ===")
print("python -m src.embeddings --config configs/default.yaml --model-path models/minilm_finetuned_main")

## 9. Measured retrieval comparison

Use the shared `src.compare` evaluator with the matching FT query encoder to produce `reports/comparison.csv`. Each system/split row records its MLflow run ID. Inspect that file and the checkpoint provenance rather than inserting a fixed result table here.
